[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/048.%20The%20Demand%20Forecasting%20-%20Exponential%20Smoothing%20Problem/P48-Tier-2_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 48. Demand Forecasting: The Exponential Smoothing Problem

## Tier 2: The Classic Heuristic — Adaptive Exponential Smoothing

### Purpose and scope

This notebook demonstrates a rule-based adaptive version of simple exponential smoothing. Given a sequence of observed weekly demand, the method produces one-step-ahead forecasts, tracks recent absolute forecast errors with a rolling mean absolute deviation (MAD), and adjusts the smoothing parameter \(\alpha\) for subsequent forecasts. The example compares this adaptive rule with simple exponential smoothing using a fixed \(\alpha\), then explores how different adjustment settings affect the same demand series.

The method is a forecasting heuristic, not a guarantee that adaptive forecasts will outperform fixed-parameter forecasts. The example uses a specified demand series and specified thresholds; results describe this data and rule, not every forecasting application.

### Background: simple exponential smoothing

A forecast combines the latest observed demand with the previous forecast. The smoothing parameter \(\alpha\) determines how much weight is given to the latest observation: a larger value responds more strongly to recent changes, while a smaller value smooths fluctuations more heavily. For demand \(D_t\) and forecast \(F_t\), the one-step update is

$$F_{t+1}=\alpha_t D_t+(1-\alpha_t)F_t.$$

Here, \(t\) indexes periods and \(\alpha_t\in[0,1]\) is the smoothing parameter used to produce the forecast for period \(t+1\). The first forecast is initialized to the first observed demand. In the static comparison, \(\alpha_t\) is fixed at 0.3. In the adaptive method, the value used for future forecasts can change as errors are observed.

### Adaptive rule and mathematical definition

A forecast error is the observed demand minus its forecast, \(e_t=D_t-F_t\). The adaptive rule monitors the absolute errors, because it uses error magnitude rather than error direction to decide whether to adjust \(\alpha\). For a rolling window of width \(w\), the recent MAD is

$$\operatorname{MAD}_t=\frac{1}{|W_t|}\sum_{i\in W_t}|e_i|,$$

where \(W_t\) contains the most recent observed absolute errors, up to \(w\) values. The adjustment rule is

$$\alpha_{t+1}=\begin{cases}
\min(\alpha_t+\Delta_{\uparrow},\alpha_{\max}), & \operatorname{MAD}_t>\tau_{\uparrow},\\
\max(\alpha_t-\Delta_{\downarrow},\alpha_{\min}), & \operatorname{MAD}_t<\tau_{\downarrow},\\
\alpha_t, & \text{otherwise},
\end{cases}$$

The symbols \(\tau_{\uparrow}\) and \(\tau_{\downarrow}\) are the error thresholds for increasing and decreasing \(\alpha\); \(\Delta_{\uparrow}\) and \(\Delta_{\downarrow}\) are the adjustment sizes; and \(\alpha_{\min}\), \(\alpha_{\max}\) bound the parameter. Thresholds and MAD use the same demand units (TEU in this example). Equality to a threshold leaves \(\alpha\) unchanged. Since the adjustment follows observation of the period's demand, it affects subsequent forecasts, not the forecast that has already been evaluated.

In the code, `adaptive_exponential_smoothing` implements the forecast-and-feedback sequence, `error_window` stores the recent absolute errors, `calculate_mad` computes their mean, and `adjust_alpha_based_on_performance` applies the bounded rule. The histories record the MAD and the value of `alpha` available after each period's observation; that value is available for later forecasts.

### Modeling assumptions

- Demand observations are finite, nonnegative values measured consistently in TEU.
- This is one-step-ahead simple exponential smoothing: it has no explicit trend, seasonal component, or forecast-uncertainty model.
- The rule treats a high recent MAD as a reason to increase responsiveness and a low recent MAD as a reason to decrease it. Thresholds and adjustment sizes are specified inputs, not learned from the data.
- MAD is calculated from absolute errors in a rolling window. It is an error-magnitude signal, not a measure of signed bias.
- Forecast accuracy metrics exclude the initialized first period because its forecast is a starting value rather than a forecast generated from a preceding observation.

### Example data and parameters

The example contains 12 weeks of port-container-terminal demand, with an increase beginning in week 7:

- Weeks 1–6: 25,000, 26,000, 24,500, 25,200, 24,800, and 25,400 TEU.
- Weeks 7–12: 32,000, 31,500, 33,200, 32,800, 33,500, and 32,200 TEU.
- The balanced adaptive rule starts at \(\alpha=0.3\), uses a 5-error window, and compares MAD with thresholds of 2,000 and 1,000 TEU.

### Learning sequence

1. Implement and inspect the adaptive forecast, error, MAD, and parameter histories.
2. Compare adaptive forecasts with a fixed-\(\alpha\) baseline using the same demand observations.
3. Visualize demand, forecasts, parameter changes, and errors.
4. Examine conservative, balanced, and aggressive settings to understand the responsiveness–stability trade-off.

### Computational cost

For \(n\) observations and a window of width \(w\), the forecast loop performs \(n-1\) updates. Each MAD calculation sums at most \(w\) absolute errors, so the loop takes \(O(nw)\) time and stores \(O(n+w)\) values when forecast and history lists are retained. If \(w\) is treated as a fixed operational setting, the runtime grows linearly with the number of observations. The implementation uses a deque to keep the rolling error window bounded.

### Interpreting the results

Use the printed metrics and plots to assess how this rule behaves on the given series. A lower error metric indicates a smaller error under that particular measure and dataset; it does not establish general superiority. Demand changes can produce large errors before the adaptive feedback has enough observations to respond. The later what-if analysis illustrates the consequences of changing the rule's settings; it does not identify universally optimal settings.

In [ ]:
# NumPy supplies numerical summaries; Matplotlib and Seaborn support the visual analysis.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Any, Dict, List, Tuple
from collections import deque
from matplotlib.patches import Patch

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
class AdaptiveExponentialSmoothing:
    """Simple exponential smoothing with a rolling-MAD adjustment rule."""

    def __init__(
        self,
        initial_alpha: float = 0.3,
        mad_window: int = 5,
        alpha_increase_threshold: float = 2000,
        alpha_decrease_threshold: float = 1000,
        alpha_adjustment_up: float = 0.1,
        alpha_adjustment_down: float = 0.05,
        min_alpha: float = 0.05,
        max_alpha: float = 0.95,
    ) -> None:
        """Store the rule's settings and initialize its forecasting state.

        Thresholds are expressed in the same units as demand. A new run
        starts from ``initial_alpha`` and an empty rolling error window.
        """
        if mad_window < 1:
            raise ValueError("mad_window must be at least 1.")
        if not 0 <= min_alpha <= initial_alpha <= max_alpha <= 1:
            raise ValueError(
                "Alpha values must satisfy 0 <= min_alpha <= "
                "initial_alpha <= max_alpha <= 1."
            )
        if alpha_decrease_threshold > alpha_increase_threshold:
            raise ValueError(
                "The decrease threshold must not exceed the increase threshold."
            )
        if alpha_adjustment_up < 0 or alpha_adjustment_down < 0:
            raise ValueError("Alpha adjustment sizes must be nonnegative.")

        self.initial_alpha = initial_alpha
        self.alpha = initial_alpha
        self.mad_window = mad_window
        self.alpha_increase_threshold = alpha_increase_threshold
        self.alpha_decrease_threshold = alpha_decrease_threshold
        self.alpha_adjustment_up = alpha_adjustment_up
        self.alpha_adjustment_down = alpha_adjustment_down
        self.min_alpha = min_alpha
        self.max_alpha = max_alpha

        self.forecasts: List[float] = []
        self.errors: List[float] = []
        self.alpha_history: List[float] = []
        self.mad_history: List[float] = []
        self.error_window = deque(maxlen=mad_window)

    def calculate_mad(self, errors: List[float]) -> float:
        """Return the mean absolute value of the supplied errors."""
        if not errors:
            return 0.0
        return sum(abs(error) for error in errors) / len(errors)

    def adjust_alpha_based_on_performance(self, recent_mad: float) -> float:
        """Apply one bounded alpha adjustment using the recent MAD."""
        if recent_mad > self.alpha_increase_threshold:
            self.alpha = min(
                self.alpha + self.alpha_adjustment_up,
                self.max_alpha,
            )
        elif recent_mad < self.alpha_decrease_threshold:
            self.alpha = max(
                self.alpha - self.alpha_adjustment_down,
                self.min_alpha,
            )
        return self.alpha

    def adaptive_exponential_smoothing(
        self, demand: List[float]
    ) -> Tuple[List[float], List[float], List[float]]:
        """Forecast demand sequentially and adapt alpha after each observation.

        Each forecast for period t+1 uses the current alpha and the
        observation from period t. Once demand for t+1 is observed, its
        absolute forecast error updates the rolling MAD and alpha for
        subsequent forecasts.
        """
        if not demand:
            raise ValueError("demand must contain at least one observation.")
        if not all(np.isfinite(value) and value >= 0 for value in demand):
            raise ValueError("Demand values must be finite and nonnegative.")

        # Reset run-specific state so repeated experiments start identically.
        self.alpha = self.initial_alpha
        self.error_window = deque(maxlen=self.mad_window)
        self.errors = []

        forecasts = [float(demand[0])]
        self.alpha_history = [self.alpha]
        self.mad_history = [0.0]

        for t in range(len(demand) - 1):
            next_forecast = (
                self.alpha * demand[t]
                + (1 - self.alpha) * forecasts[t]
            )
            forecasts.append(next_forecast)

            # The new observation evaluates this forecast and informs the next update.
            error = demand[t + 1] - next_forecast
            self.errors.append(error)
            self.error_window.append(abs(error))
            recent_mad = self.calculate_mad(list(self.error_window))
            self.mad_history.append(recent_mad)
            self.adjust_alpha_based_on_performance(recent_mad)
            self.alpha_history.append(self.alpha)

        self.forecasts = forecasts
        return forecasts, self.alpha_history, self.mad_history

    def calculate_forecast_errors(
        self, demand: List[float], forecasts: List[float]
    ) -> List[float]:
        """Return signed errors, defined as observed demand minus forecast."""
        if len(demand) != len(forecasts):
            raise ValueError("demand and forecasts must have equal lengths.")
        return [actual - forecast for actual, forecast in zip(demand, forecasts)]

    def calculate_accuracy_metrics(
        self, demand: List[float], forecasts: List[float]
    ) -> Dict[str, float]:
        """Calculate MSE, MAE, MAPE, and RMSE after the initial period."""
        errors = self.calculate_forecast_errors(demand, forecasts)
        valid_errors = errors[1:]

        if not valid_errors:
            return {"mse": 0.0, "mae": 0.0, "mape": 0.0, "rmse": 0.0}

        mse = sum(error ** 2 for error in valid_errors) / len(valid_errors)
        mae = sum(abs(error) for error in valid_errors) / len(valid_errors)
        rmse = float(np.sqrt(mse))
        percentage_errors = [
            abs(error) / demand[index] * 100
            for index, error in enumerate(errors[1:], start=1)
            if demand[index] != 0
        ]
        mape = (
            sum(percentage_errors) / len(percentage_errors)
            if percentage_errors
            else 0.0
        )

        return {"mse": mse, "mae": mae, "mape": mape, "rmse": rmse}

    def compare_with_static(
        self, demand: List[float], static_alpha: float = 0.3
    ) -> Dict[str, Any]:
        """Compare the adaptive rule with fixed-alpha simple smoothing."""
        if not 0 <= static_alpha <= 1:
            raise ValueError("static_alpha must be between 0 and 1.")

        adaptive_forecasts, adaptive_alphas, adaptive_mads = (
            self.adaptive_exponential_smoothing(demand)
        )
        adaptive_metrics = self.calculate_accuracy_metrics(
            demand, adaptive_forecasts
        )

        if not demand:
            raise ValueError("demand must contain at least one observation.")
        static_forecasts = [float(demand[0])]
        for t in range(len(demand) - 1):
            next_forecast = (
                static_alpha * demand[t]
                + (1 - static_alpha) * static_forecasts[t]
            )
            static_forecasts.append(next_forecast)

        static_metrics = self.calculate_accuracy_metrics(demand, static_forecasts)
        return {
            "adaptive": {
                "forecasts": adaptive_forecasts,
                "alphas": adaptive_alphas,
                "mads": adaptive_mads,
                "metrics": adaptive_metrics,
            },
            "static": {
                "forecasts": static_forecasts,
                "alpha": static_alpha,
                "metrics": static_metrics,
            },
        }

## The Port-Terminal Demand Example

The next cell applies the specified balanced rule to the 12 observations. The forecast for each week is formed using information available before that week's demand is observed. After the demand is revealed, its error enters the rolling window and can change alpha for later forecasts. The first forecast is initialized from week 1 demand, so it is shown in the table but excluded from the accuracy metrics.

In [ ]:
# Weekly container demand, measured in twenty-foot equivalent units (TEU).
extended_demand = [
    25000, 26000, 24500, 25200, 24800, 25400,
    32000, 31500, 33200, 32800, 33500, 32200,
]
weeks = list(range(1, len(extended_demand) + 1))

print("=== ADAPTIVE EXPONENTIAL SMOOTHING: PORT-TERMINAL EXAMPLE ===")
print(f"Weekly demand (TEU): {extended_demand}")
print(f"Number of periods: {len(extended_demand)}")
print("Demand increase begins in week 7 (from 25,400 to 32,000 TEU).")

adaptive_es = AdaptiveExponentialSmoothing(
    initial_alpha=0.3,
    mad_window=5,
    alpha_increase_threshold=2000,
    alpha_decrease_threshold=1000,
    alpha_adjustment_up=0.1,
    alpha_adjustment_down=0.05,
    min_alpha=0.05,
    max_alpha=0.95,
)

adaptive_forecasts, adaptive_alphas, adaptive_mads = (
    adaptive_es.adaptive_exponential_smoothing(extended_demand)
)
adaptive_errors = adaptive_es.calculate_forecast_errors(
    extended_demand, adaptive_forecasts
)
adaptive_metrics = adaptive_es.calculate_accuracy_metrics(
    extended_demand, adaptive_forecasts
)

print("\n=== ADAPTIVE METHOD RESULTS ===")
print("Week | Demand | Forecast | Alpha after observation | Error | Rolling MAD")
print("-" * 73)
for i, (week, actual, forecast, alpha, error, mad) in enumerate(
    zip(
        weeks,
        extended_demand,
        adaptive_forecasts,
        adaptive_alphas,
        adaptive_errors,
        adaptive_mads,
    )
):
    error_text = "N/A" if i == 0 else f"{error:.0f}"
    print(
        f"{week:4d} | {actual:6d} | {forecast:8.0f} | "
        f"{alpha:21.3f} | {error_text:>6} | {mad:11.0f}"
    )

print("\n=== ADAPTIVE PERFORMANCE METRICS ===")
print(f"Mean Squared Error (TEU²): {adaptive_metrics['mse']:,.0f}")
print(f"Mean Absolute Error (TEU): {adaptive_metrics['mae']:,.0f}")
print(f"Mean Absolute Percentage Error: {adaptive_metrics['mape']:.1f}%")
print(f"Root Mean Squared Error (TEU): {adaptive_metrics['rmse']:,.0f}")
print(f"Final alpha: {adaptive_alphas[-1]:.3f}")
print(
    f"Alpha range: {min(adaptive_alphas):.3f} "
    f"to {max(adaptive_alphas):.3f}"
)

## Comparison with Fixed-Parameter Smoothing

Both methods use the same initialization and demand observations. The fixed-parameter baseline uses \(\alpha=0.3\) throughout; the adaptive method changes alpha only after observing a forecast error. MSE, MAE, and RMSE are expressed in TEU-based units, while MAPE is a percentage. The reported improvement is calculated as \((\text{static error}-\text{adaptive error})/\text{static error}\times100\%\); a positive value means the adaptive metric is lower for this series, and a negative value means it is higher. These comparisons are descriptive, not evidence of performance on other data.

In [ ]:
print("=== ADAPTIVE VS FIXED-ALPHA EXPONENTIAL SMOOTHING ===")
comparison = adaptive_es.compare_with_static(extended_demand, static_alpha=0.3)
static_forecasts = comparison["static"]["forecasts"]
static_metrics = comparison["static"]["metrics"]
static_errors = adaptive_es.calculate_forecast_errors(
    extended_demand, static_forecasts
)

print("\n--- PERFORMANCE COMPARISON ---")
print(f"{'Metric':<15} | {'Fixed alpha':>13} | {'Adaptive':>13} | {'Change':>10}")
print("-" * 62)
metrics_comparison = [
    ("MSE (TEU²)", static_metrics["mse"], adaptive_metrics["mse"]),
    ("MAE (TEU)", static_metrics["mae"], adaptive_metrics["mae"]),
    ("MAPE (%)", static_metrics["mape"], adaptive_metrics["mape"]),
    ("RMSE (TEU)", static_metrics["rmse"], adaptive_metrics["rmse"]),
]

for metric_name, static_value, adaptive_value in metrics_comparison:
    if static_value > 0:
        change = (static_value - adaptive_value) / static_value * 100
        change_text = f"{change:+.1f}%"
    else:
        change_text = "N/A"
    print(
        f"{metric_name:<15} | {static_value:13.1f} | "
        f"{adaptive_value:13.1f} | {change_text:>10}"
    )

print("\n--- PERIOD-BY-PERIOD COMPARISON ---")
print("Week | Demand | Fixed forecast | Adaptive forecast | Fixed error | Adaptive error | Alpha")
print("-" * 89)
for i, week in enumerate(weeks):
    if i == 0:
        fixed_error_text = "N/A"
        adaptive_error_text = "N/A"
    else:
        fixed_error_text = f"{static_errors[i]:.0f}"
        adaptive_error_text = f"{adaptive_errors[i]:.0f}"
    print(
        f"{week:4d} | {extended_demand[i]:6d} | "
        f"{static_forecasts[i]:14.0f} | {adaptive_forecasts[i]:17.0f} | "
        f"{fixed_error_text:>11} | {adaptive_error_text:>14} | "
        f"{adaptive_alphas[i]:.3f}"
    )

print("\n--- RESPONSE AROUND THE DEMAND INCREASE ---")
print(f"Mean alpha, weeks 1–6: {np.mean(adaptive_alphas[:6]):.3f}")
print(f"Alpha standard deviation, weeks 1–6: {np.std(adaptive_alphas[:6]):.3f}")
print(f"Mean rolling MAD, weeks 1–6: {np.mean(adaptive_mads[:6]):.0f} TEU")
print(f"Mean alpha, weeks 7–12: {np.mean(adaptive_alphas[6:]):.3f}")
print(f"Alpha standard deviation, weeks 7–12: {np.std(adaptive_alphas[6:]):.3f}")
print(f"Mean rolling MAD, weeks 7–12: {np.mean(adaptive_mads[6:]):.0f} TEU")
print(
    f"Alpha change recorded after week 7: "
    f"{adaptive_alphas[6] - adaptive_alphas[5]:+.3f}"
)
post_shift_peak = max(adaptive_alphas[6:])
post_shift_peak_week = 7 + adaptive_alphas[6:].index(post_shift_peak)
print(f"Peak alpha in weeks 7–12: {post_shift_peak:.3f} (week {post_shift_peak_week})")
print(f"Alpha recorded after week 12: {adaptive_alphas[-1]:.3f}")

## Visualizing Forecasts, Errors, and Adaptation

The first dashboard compares demand and forecasts, then shows alpha, rolling MAD, and signed errors over time. The demand-shift marker lies between weeks 6 and 7. Remember that the alpha history is recorded after each week's demand and error are observed; an adjustment at week 7 is available to influence a later forecast, not the forecast for week 7 itself. The second dashboard summarizes cumulative absolute error, metrics, alpha adjustments, and the distributions of signed errors.

In [ ]:
# Dashboard 1: forecasts and the feedback signals that drive adaptation.
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Adaptive Exponential Smoothing: Forecast and Feedback Analysis",
    fontsize=16,
    fontweight="bold",
)

ax1.plot(weeks, extended_demand, "o-", linewidth=3, markersize=7,
         label="Actual demand", color="black")
ax1.plot(weeks, static_forecasts, "s--", linewidth=2, markersize=5,
         label="Fixed alpha = 0.3", color="red")
ax1.plot(weeks, adaptive_forecasts, "^:", linewidth=2, markersize=5,
         label="Adaptive smoothing", color="green")
ax1.axvline(x=6.5, color="orange", linestyle="--", linewidth=2,
            alpha=0.7, label="Demand increase")
ax1.set_xlabel("Week")
ax1.set_ylabel("Container demand (TEU)")
ax1.set_title("Observed Demand and One-Step-Ahead Forecasts")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(weeks, adaptive_alphas, "o-", linewidth=2, markersize=5,
         color="blue", label="Adaptive alpha after observation")
ax2.axhline(y=0.3, color="red", linestyle="--", linewidth=2,
            label="Fixed alpha = 0.3")
ax2.axvline(x=6.5, color="orange", linestyle="--", linewidth=2,
            alpha=0.7, label="Demand increase")
ax2.set_xlabel("Week")
ax2.set_ylabel("Smoothing parameter, alpha")
ax2.set_title("Alpha Available for Subsequent Forecasts")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3.plot(weeks[1:], adaptive_mads[1:], "s-", linewidth=2, markersize=5,
         color="purple", label="Rolling MAD (up to 5 errors)")
ax3.axhline(y=2000, color="red", linestyle="--", linewidth=1,
            alpha=0.7, label="Alpha-increase threshold")
ax3.axhline(y=1000, color="green", linestyle="--", linewidth=1,
            alpha=0.7, label="Alpha-decrease threshold")
ax3.axvline(x=6.5, color="orange", linestyle="--", linewidth=2,
            alpha=0.7, label="Demand increase")
ax3.set_xlabel("Week")
ax3.set_ylabel("Mean absolute deviation (TEU)")
ax3.set_title("Recent Absolute Error and Adjustment Thresholds")
ax3.legend()
ax3.grid(True, alpha=0.3)

valid_weeks = weeks[1:]
static_valid_errors = static_errors[1:]
adaptive_valid_errors = adaptive_errors[1:]
ax4.plot(valid_weeks, static_valid_errors, "s-", linewidth=2,
         markersize=5, label="Fixed-alpha errors", color="red")
ax4.plot(valid_weeks, adaptive_valid_errors, "o-", linewidth=2,
         markersize=5, label="Adaptive errors", color="green")
ax4.axhline(y=0, color="black", linestyle="-", linewidth=0.7)
ax4.axvline(x=6.5, color="orange", linestyle="--", linewidth=2,
            alpha=0.7, label="Demand increase")
ax4.set_xlabel("Week")
ax4.set_ylabel("Forecast error (TEU; actual minus forecast)")
ax4.set_title("Signed Forecast Errors")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Dashboard 2: cumulative error, metric scales, parameter changes, and error distributions.
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Adaptive Exponential Smoothing: Performance Dashboard",
    fontsize=16,
    fontweight="bold",
)

cumulative_static_error = np.cumsum(np.abs(static_valid_errors))
cumulative_adaptive_error = np.cumsum(np.abs(adaptive_valid_errors))
ax1.plot(valid_weeks, cumulative_static_error, "s-", linewidth=2,
         markersize=5, label="Fixed alpha", color="red")
ax1.plot(valid_weeks, cumulative_adaptive_error, "o-", linewidth=2,
         markersize=5, label="Adaptive", color="green")
ax1.axvline(x=6.5, color="orange", linestyle="--", linewidth=2,
            alpha=0.7, label="Demand increase")
ax1.set_xlabel("Week")
ax1.set_ylabel("Cumulative absolute error (TEU)")
ax1.set_title("Cumulative Absolute Error")
ax1.legend()
ax1.grid(True, alpha=0.3)

metric_names = ["MSE", "MAE", "MAPE", "RMSE"]
static_values = [
    static_metrics["mse"] / 1000,
    static_metrics["mae"],
    static_metrics["mape"],
    static_metrics["rmse"],
]
adaptive_values = [
    adaptive_metrics["mse"] / 1000,
    adaptive_metrics["mae"],
    adaptive_metrics["mape"],
    adaptive_metrics["rmse"],
]
x = np.arange(len(metric_names))
width = 0.35
ax2.bar(x - width / 2, static_values, width, label="Fixed alpha",
        color="red", alpha=0.7)
ax2.bar(x + width / 2, adaptive_values, width, label="Adaptive",
        color="green", alpha=0.7)
ax2.set_xlabel("Metric")
ax2.set_ylabel("Metric value (MSE divided by 1,000)")
ax2.set_title("Metrics (MSE in thousands of TEU²; MAPE in percent)")
ax2.set_xticks(x)
ax2.set_xticklabels(metric_names)
ax2.legend()
ax2.grid(True, alpha=0.3)

alpha_changes = np.diff(adaptive_alphas)
change_weeks = weeks[1:]
change_colors = [
    "green" if change > 0 else "red" if change < 0 else "blue"
    for change in alpha_changes
]
ax3.bar(change_weeks, alpha_changes, color=change_colors, alpha=0.7)
ax3.axhline(y=0, color="black", linestyle="-", linewidth=0.7)
ax3.axvline(x=6.5, color="orange", linestyle="--", linewidth=2,
            alpha=0.7, label="Demand increase")
ax3.set_xlabel("Week of observation")
ax3.set_ylabel("Change in alpha after observation")
ax3.set_title("Parameter Adjustments")
ax3.legend(handles=[
    Patch(facecolor="green", alpha=0.7, label="Increase"),
    Patch(facecolor="red", alpha=0.7, label="Decrease"),
    Patch(facecolor="blue", alpha=0.7, label="No change"),
    Patch(facecolor="orange", alpha=0.7, label="Demand increase marker"),
])
ax3.grid(True, alpha=0.3)

ax4.hist(static_valid_errors, bins=8, alpha=0.65, label="Fixed alpha",
         color="red", density=True)
ax4.hist(adaptive_valid_errors, bins=8, alpha=0.65, label="Adaptive",
         color="green", density=True)
ax4.set_xlabel("Forecast error (TEU; actual minus forecast)")
ax4.set_ylabel("Probability density (1/TEU)")
ax4.set_title("Distribution of Signed Forecast Errors")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## What-If Analysis: Alternative Adjustment Settings

This experiment keeps the demand observations and the rule's structure fixed while varying its settings. Conservative settings use smaller adjustments and a wider separation between the thresholds; aggressive settings use a shorter error window, different thresholds, and larger adjustments. The balanced model uses the example's original settings. Each model run begins at its configured initial alpha with an empty rolling window, so the strategies are compared from the same starting conditions. Differences illustrate a responsiveness–stability trade-off on this series; they do not prove that any strategy is best in general.

In [ ]:
print("=== WHAT-IF ANALYSIS: ADAPTATION SETTINGS ===")

conservative_es = AdaptiveExponentialSmoothing(
    initial_alpha=0.3,
    mad_window=5,
    alpha_increase_threshold=2000,
    alpha_decrease_threshold=1000,
    alpha_adjustment_up=0.05,
    alpha_adjustment_down=0.025,
    min_alpha=0.1,
    max_alpha=0.8,
)

aggressive_es = AdaptiveExponentialSmoothing(
    initial_alpha=0.3,
    mad_window=3,
    alpha_increase_threshold=1500,
    alpha_decrease_threshold=800,
    alpha_adjustment_up=0.2,
    alpha_adjustment_down=0.15,
    min_alpha=0.05,
    max_alpha=0.95,
)

strategies = [
    ("Conservative", conservative_es),
    ("Balanced", adaptive_es),
    ("Aggressive", aggressive_es),
]
strategy_results = []

for strategy_name, strategy_model in strategies:
    forecasts, alphas, mads = strategy_model.adaptive_exponential_smoothing(
        extended_demand
    )
    metrics = strategy_model.calculate_accuracy_metrics(
        extended_demand, forecasts
    )
    strategy_results.append(
        {
            "strategy": strategy_name,
            "forecasts": forecasts,
            "alphas": alphas,
            "mads": mads,
            "metrics": metrics,
            "alpha_range": max(alphas) - min(alphas),
            "alpha_volatility": float(np.std(alphas)),
        }
    )

print("\n--- STRATEGY COMPARISON ---")
print(
    f"{'Strategy':<13} | {'MSE (TEU²)':>12} | {'MAE (TEU)':>10} | "
    f"{'MAPE (%)':>9} | {'Alpha range':>12} | {'Alpha SD':>9}"
)
print("-" * 83)
for result in strategy_results:
    metrics = result["metrics"]
    print(
        f"{result['strategy']:<13} | {metrics['mse']:12.1f} | "
        f"{metrics['mae']:10.1f} | {metrics['mape']:9.2f} | "
        f"{result['alpha_range']:12.3f} | "
        f"{result['alpha_volatility']:9.3f}"
    )

print("\n--- ALPHA RESPONSE BY STRATEGY ---")
for result in strategy_results:
    alphas = result["alphas"]
    pre_shift_alpha = float(np.mean(alphas[:6]))
    post_shift_alpha = float(np.mean(alphas[6:]))
    peak_alpha = max(alphas)
    peak_week = alphas.index(peak_alpha) + 1
    print(f"\n{result['strategy']} settings:")
    print(f"  Mean alpha, weeks 1–6: {pre_shift_alpha:.3f}")
    print(f"  Mean alpha, weeks 7–12: {post_shift_alpha:.3f}")
    print(f"  Difference between period means: {post_shift_alpha - pre_shift_alpha:+.3f}")
    print(f"  Peak recorded alpha: {peak_alpha:.3f} (week {peak_week})")
    print(f"  Alpha standard deviation: {result['alpha_volatility']:.3f}")

print("\n=== INTERPRETATION ===")
print(
    "Smaller adjustments generally limit the size of each parameter change; "
    "larger adjustments can change responsiveness more sharply."
)
print(
    "The window width and thresholds also affect when the rule responds. "
    "Interpret each metric together with the alpha history and forecast errors."
)
print(
    "A strategy's relative performance on this example is not a guarantee "
    "of performance on future or different demand series."
)

## Summary and Conclusions

### What the adaptive heuristic does

Adaptive exponential smoothing combines the simple exponential smoothing update with a feedback rule. After each forecast is evaluated, the method appends the absolute error to a bounded rolling window, calculates its MAD, and adjusts alpha according to the two thresholds. Bounds keep alpha within the configured interval. The next forecast uses the updated parameter; the forecast for an already observed period is not revised.

### How to read the example

The printed results and figures show the forecasts, signed errors, rolling MAD, alpha history, and accuracy metrics for the supplied 12-week series. Compare the fixed and adaptive methods using the same periods and metric definitions. The what-if results show how the chosen settings affect this particular run. A larger alpha can make forecasts more responsive to recent observations, but abrupt changes in demand can still cause errors, and reacting strongly to random fluctuations can reduce stability. No single metric or short example establishes general forecasting superiority.

### Practical considerations and limitations

Thresholds should be chosen in the same units as demand and calibrated to the scale and operating context of the series. Window width controls how many recent absolute errors contribute to MAD. Alpha bounds and adjustment sizes control the permitted parameter movement. Because the rule reacts to error magnitude, it does not distinguish systematic over-forecasting from under-forecasting. The method also has no explicit trend or seasonal components, learns no rule from historical patterns, produces only one-step-ahead forecasts, and does not quantify forecast uncertainty.

### Computational considerations

For \(n\) observations and window width \(w\), this implementation takes \(O(nw)\) time because it recomputes the mean of up to \(w\) absolute errors at each update. With forecast and diagnostic histories retained, its storage is \(O(n+w)\). When the chosen window width is fixed, runtime is linear in the number of observations. The rolling window bounds the amount of error history needed for each MAD calculation.

### Further study

A natural next step in the sequence is Tier 3: Genetic Algorithm for Multi-Parameter Optimization. That method addresses a broader parameter-search problem, including trend and seasonal components; it is distinct from the rule-based adaptive heuristic implemented here.